# 讀取資料

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings
warnings.simplefilter('ignore')

pd.set_option('display.max_columns', 100)

TARGET = 'Listening_Time_minutes'
CATS = ['Podcast_Name', 'Episode_Num', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']
NUMS = ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads']

oof_pred_name = '2+3+4_interact_xgb'

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s5e4/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s5e4/test.csv', index_col='id')
original = pd.read_csv('/kaggle/input/podcast-listening-time-prediction-dataset/podcast_dataset.csv')
print(f"Train shape: {train.shape}")
print(f"Test  shape: {test.shape}")
print(f"Orig  shape: {original.shape}")
train.head()

In [ ]:
# 移除異常值
train['Episode_Length_minutes'] = np.maximum(0, np.minimum(120, train['Episode_Length_minutes']))
train['Host_Popularity_percentage'] = np.maximum(20, np.minimum(100, train['Host_Popularity_percentage']))
train['Guest_Popularity_percentage'] = np.maximum(0, np.minimum(100, train['Guest_Popularity_percentage']))
train.loc[train['Number_of_Ads'] > 3, 'Number_of_Ads'] = 0

In [ ]:
original_clean = original.dropna(subset=[TARGET]).drop_duplicates()
train = pd.concat([train, original_clean], axis=0, ignore_index=True)

In [ ]:
train.info()

In [ ]:
# 缺失值填補
train[NUMS] = train[NUMS].fillna(train[NUMS].median())
test[NUMS] = test[NUMS].fillna(train[NUMS].median())

# 特徵工程

In [ ]:
def feature_eng(df):

    df = df.copy()

    # 擷取集數
    df['Episode_Num'] = pd.to_numeric(df['Episode_Title'].astype(str).str[8:], errors='coerce')

    # 判斷是否為週末
    df['is_weekend'] = df['Publication_Day'].isin(['Saturday', 'Sunday']).astype(int)

    # 平方與開根號
    col = 'Episode_Length_minutes'
    df[col + '_sqrt'] = np.sqrt(df[col].clip(lower=0))
    df[col + '_squared'] = df[col] ** 2

    return df

# 類別資料轉換
day_mapping = {
    'Monday': 1, 'Tuesday': 2, 'Wednesday': 3,
    'Thursday': 4, 'Friday': 5,
    'Saturday': 6, 'Sunday': 7}

time_mapping = {
    'Morning': 1, 'Afternoon': 2,
    'Evening': 3, 'Night': 4}

sentiment_mapping = {
    'Negative': 1, 'Neutral': 2, 'Positive': 3}

def encode_features(df):

    df = df.copy()

    df['Publication_Day'] = df['Publication_Day'].map(day_mapping)
    df['Publication_Time'] = df['Publication_Time'].map(time_mapping)
    df['Episode_Sentiment'] = df['Episode_Sentiment'].map(sentiment_mapping)

    return df

def add_group_features(train_df, test_df):

    train_df = train_df.copy()
    test_df = test_df.copy()

    GROUP_COLS = [
        'Genre',
        'Podcast_Name',
        'Publication_Day',
        'Publication_Time',
        'Episode_Sentiment']

    AGG_COLS = [
        'mean',
        'median',
        'std',
        'count',
        'min',
        'max']

    for col in GROUP_COLS:

        group_stats = train_df.groupby(col)[
            'Episode_Length_minutes'].agg(AGG_COLS)

        for stat in AGG_COLS:

            feature_name = f'{col}_ELM_{stat}'

            mapping = group_stats[stat]

            train_df[feature_name] = train_df[col].map(mapping)
            test_df[feature_name] = test_df[col].map(mapping)

    return train_df, test_df

def add_interaction_features(df):

    df = df.copy()

    # 兩兩交互特徵
    selected_comb = [
        ['Episode_Length_minutes', 'Host_Popularity_percentage'],
        ['Episode_Length_minutes', 'Guest_Popularity_percentage'],
        ['Episode_Length_minutes', 'Number_of_Ads'],
        ['Episode_Num', 'Host_Popularity_percentage'],
        ['Episode_Num', 'Guest_Popularity_percentage'],
        ['Episode_Num', 'Number_of_Ads'],
        ['Host_Popularity_percentage', 'Guest_Popularity_percentage'],
        ['Host_Popularity_percentage', 'Number_of_Ads'],
        ['Guest_Popularity_percentage', 'Number_of_Ads']]

    for col1, col2 in selected_comb:

        df[f'{col1}_x_{col2}'] = df[col1] * df[col2]

    return df

def add_ratio_features(df):

    df = df.copy()

    # 每分鐘廣告數量
    df['Ads_Per_Minute'] = (
        df['Number_of_Ads'] /
        df['Episode_Length_minutes'].clip(lower=1))

    # 主持人與來賓知名度差距
    df['Popularity_Difference'] = (
        df['Host_Popularity_percentage'] -
        df['Guest_Popularity_percentage'])

    # 主持人與來賓平均知名度
    df['Popularity_Mean'] = (
        df['Host_Popularity_percentage'] +
        df['Guest_Popularity_percentage']) / 2

    # 最高知名度
    df['Popularity_Max'] = df[
        ['Host_Popularity_percentage',
         'Guest_Popularity_percentage']].max(axis=1)

    # 最低知名度
    df['Popularity_Min'] = df[
        ['Host_Popularity_percentage',
         'Guest_Popularity_percentage']].min(axis=1)

    # 廣告數量與節目長度的比例
    df['Ads_Length_Ratio'] = (
        df['Number_of_Ads'] /
        df['Episode_Length_minutes'].clip(lower=1))

    return df

def add_cyclical_features(df):

    df = df.copy()

    # 星期週期
    df['Day_Sin'] = np.sin(2 * np.pi * (df['Publication_Day'] - 1) / 7)
    df['Day_Cos'] = np.cos(2 * np.pi * (df['Publication_Day'] - 1) / 7)

    # 發布時段週期
    df['Time_Sin'] = np.sin(2 * np.pi * (df['Publication_Time'] - 1) / 4)
    df['Time_Cos'] = np.cos(2 * np.pi * (df['Publication_Time'] - 1) / 4)

    return df

def add_episode_features(df):

    df = df.copy()

    # 集數開根號
    df['Episode_Num_sqrt'] = np.sqrt(df['Episode_Num'].clip(lower=0))

    # 集數平方
    df['Episode_Num_squared'] = (df['Episode_Num'] ** 2)

    # 節目發展階段
    df['Episode_Stage'] = pd.cut(
        df['Episode_Num'],
        bins=[-1, 10, 50, 100, np.inf],
        labels=[1, 2, 3, 4]).astype(float)

    return df

# 1. 原始特徵工程
train = feature_eng(train)
test = feature_eng(test)

# 2. 類別轉換
train = encode_features(train)
test = encode_features(test)

# 3. 集數衍生特徵
train = add_episode_features(train)
test = add_episode_features(test)

# 4. 交互特徵
train = add_interaction_features(train)
test = add_interaction_features(test)

# 5. 比例特徵
train = add_ratio_features(train)
test = add_ratio_features(test)

# 6. 週期特徵
train = add_cyclical_features(train)
test = add_cyclical_features(test)

# 7. 群組統計特徵
train, test = add_group_features(train, test)

# 8. 移除不再使用的欄位
train.drop(columns=['Episode_Title'], inplace=True)
test.drop(columns=['Episode_Title'], inplace=True)

In [ ]:
ELM = []
for k in range(3):
    col_name = f'ELm_r{k}'
    train[col_name] = train['Episode_Length_minutes'].round(k)
    test[col_name] = test['Episode_Length_minutes'].round(k)
    ELM.append(col_name)

In [ ]:
encoded_columns = []

selected_comb = [
    # 2-interaction
    ['Episode_Length_minutes', 'Host_Popularity_percentage'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage'],
    ['Episode_Length_minutes', 'Number_of_Ads'],
    ['Episode_Num', 'Host_Popularity_percentage'],
    ['Episode_Num', 'Guest_Popularity_percentage'],
    ['Episode_Num', 'Number_of_Ads'],
    ['Host_Popularity_percentage', 'Guest_Popularity_percentage'],
    ['Host_Popularity_percentage', 'Number_of_Ads'],
    ['Host_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Podcast_Name'],
    ['Episode_Num', 'Podcast_Name'],
    ['Guest_Popularity_percentage', 'Podcast_Name'],
    ['ELm_r1', 'Episode_Num'],
    ['ELm_r1', 'Host_Popularity_percentage'],
    ['ELm_r1', 'Guest_Popularity_percentage'],
    ['ELm_r2', 'Episode_Num'],
    ['ELm_r2', 'Episode_Sentiment'],
    ['ELm_r2', 'Publication_Day'],


    # 3-interaction
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage'],
    ['Episode_Length_minutes', 'Episode_Num', 'Guest_Popularity_percentage'],
    ['Episode_Length_minutes', 'Episode_Num', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Episode_Num', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Episode_Num', 'Publication_Day'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Publication_Day'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Publication_Time'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Publication_Day'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Publication_Time'],
    ['Episode_Length_minutes', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Number_of_Ads', 'Publication_Day'],
    ['Episode_Length_minutes', 'Episode_Sentiment', 'Publication_Time'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Publication_Day'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Publication_Time'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Genre'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Publication_Day'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Publication_Time'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Genre'],
    ['Episode_Num', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Episode_Sentiment'],
    ['Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Publication_Day'],
    ['Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Publication_Time'],
    ['Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Day'],

    ['Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Guest_Popularity_percentage', 'Number_of_Ads', 'Genre'],
    ['ELm_r1', 'Number_of_Ads', 'Episode_Sentiment'],
    ['ELm_r2', 'Number_of_Ads', 'Podcast_Name'],

    # 4-interaction
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage'],
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Publication_Day'],
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Publication_Time'],
    ['Episode_Length_minutes', 'Episode_Num', 'Host_Popularity_percentage', 'Genre'],
    ['Episode_Length_minutes', 'Episode_Num', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Episode_Num', 'Guest_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Episode_Num', 'Guest_Popularity_percentage', 'Publication_Day'],
    ['Episode_Length_minutes', 'Episode_Num', 'Guest_Popularity_percentage', 'Publication_Time'],
    ['Episode_Length_minutes', 'Episode_Num', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Episode_Num', 'Number_of_Ads', 'Publication_Day'],
    ['Episode_Length_minutes', 'Episode_Num', 'Number_of_Ads', 'Publication_Time'],
    ['Episode_Length_minutes', 'Episode_Num', 'Publication_Day', 'Publication_Time'],
    ['Episode_Length_minutes', 'Episode_Num', 'Publication_Day', 'Genre'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Publication_Day'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Publication_Time'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Day'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Publication_Day', 'Publication_Time'],
    ['Episode_Length_minutes', 'Host_Popularity_percentage', 'Publication_Day', 'Genre'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Publication_Day'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Publication_Time'],
    ['Episode_Length_minutes', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Genre'],
    ['Episode_Length_minutes', 'Episode_Num', 'Publication_Time', 'Podcast_Name'],

    ['Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Number_of_Ads'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Guest_Popularity_percentage', 'Episode_Sentiment'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Day'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Time'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment', 'Publication_Day'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment', 'Publication_Time'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment', 'Genre'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Publication_Day', 'Publication_Time'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Publication_Time', 'Genre'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment'],
    ['Episode_Num', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Genre'],
    ['Episode_Num', 'Host_Popularity_percentage', 'Episode_Sentiment', 'Podcast_Name'],
    ['Host_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment', 'Podcast_Name'],
    ['Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Day', 'Podcast_Name'],
    ['Host_Popularity_percentage', 'Number_of_Ads', 'Publication_Time', 'Podcast_Name'],

]

for comb in selected_comb:
    name = '_'.join(comb)

    if len(comb) == 2:
        train[name] = train[comb[0]].astype(str) + '_' + train[comb[1]].astype(str)
        test[name] = test[comb[0]].astype(str) + '_' + test[comb[1]].astype(str)

    elif len(comb) == 3:
        train[name] = (train[comb[0]].astype(str) + '_' +
                       train[comb[1]].astype(str) + '_' +
                       train[comb[2]].astype(str))
        test[name] = (test[comb[0]].astype(str) + '_' +
                      test[comb[1]].astype(str) + '_' +
                      test[comb[2]].astype(str))

    elif len(comb) == 4:
        train[name] = (train[comb[0]].astype(str) + '_' +
                       train[comb[1]].astype(str) + '_' +
                       train[comb[2]].astype(str) + '_' +
                       train[comb[3]].astype(str))
        test[name] = (test[comb[0]].astype(str) + '_' +
                      test[comb[1]].astype(str) + '_' +
                      test[comb[2]].astype(str) + '_' +
                      test[comb[3]].astype(str))

    encoded_columns.append(name)

train[encoded_columns] = train[encoded_columns].astype('category')
test[encoded_columns] = test[encoded_columns].astype('category')

In [ ]:
def frequency_encoding(train_df, test_df, columns):

    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in columns:

        freq = train_df[col].value_counts(normalize=True)

        train_df[col + '_Frequency'] = train_df[col].map(freq)
        test_df[col + '_Frequency'] = test_df[col].map(freq).fillna(0)

    return train_df, test_df

def one_hot_encoding(train_df, test_df, columns):

    train_df = train_df.copy()
    test_df = test_df.copy()

    combined = pd.concat([train_df[columns], test_df[columns]], axis=0)

    encoded = pd.get_dummies(combined, columns=columns, dtype=int)

    train_encoded = encoded.iloc[:len(train_df)].copy()
    test_encoded = encoded.iloc[len(train_df):].copy()

    train_df = pd.concat([train_df.drop(columns=columns).reset_index(drop=True), train_encoded.reset_index(drop=True)], axis=1)
    test_df = pd.concat([test_df.drop(columns=columns).reset_index(drop=True), test_encoded.reset_index(drop=True)], axis=1)

    return train_df, test_df

train, test = frequency_encoding(train, test, ['Podcast_Name'])
train, test = one_hot_encoding(train, test, ['Genre'])

train.drop(columns=['Podcast_Name'], inplace=True)
test.drop(columns=['Podcast_Name'], inplace=True)

In [ ]:
# 建立模型特徵
TARGET = 'Listening_Time_minutes'
FEATURES = [col for col in train.columns if col != TARGET]

NUMS = train[FEATURES].select_dtypes(include=['int64', 'int32', 'float64', 'float32']).columns.tolist()
CATS = train[FEATURES].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")

print(f"Total Features: {len(FEATURES)}")
print(f"Numerical Features: {len(NUMS)}")
print(f"Categorical Features: {len(CATS)}")

print(
    f"Train RAM: "
    f"{train.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

print(
    f"Test RAM: "
    f"{test.memory_usage(deep=True).sum() / 1024**3:.2f} GB")

In [ ]:
train.info()

# Train XGBoost

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import gc

In [ ]:
def target_encode(df_train, df_val, col, target, stats='mean', prefix='TE'):
    df_val = df_val.copy()
    agg = df_train.groupby(col)[target].agg(stats)    
    if isinstance(stats, (list, tuple)):
        for s in stats:
            colname = f"{prefix}_{col}_{s}"
            df_val[colname] = df_val[col].map(agg[s]).astype(float)
            df_val[colname].fillna(agg[s].mean(), inplace=True)
    else:
        suffix = stats if isinstance(stats, str) else stats.__name__
        colname = f"{prefix}_{col}_{suffix}"
        df_val[colname] = df_val[col].map(agg).astype(float)
        df_val[colname].fillna(agg.mean(), inplace=True)
    return df_val

In [ ]:
# reference: https://www.kaggle.com/code/act18l/say-goodbye-to-ordinalencoder
class OrderedTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Out‑of‑fold **mean‑rank** encoder with optional smoothing.
    • Encodes each category by the *rank* of its target mean within a fold.
    • Unseen categories get the global mean rank (or −1 if you prefer).
    """
    def __init__(self, cat_cols=None, n_splits=5, smoothing=0):
        self.cat_cols   = cat_cols
        self.n_splits   = n_splits
        self.smoothing  = smoothing       # 0 = no smoothing
        self.maps_      = {}              # per‑fold maps
        self.global_map = {}              # fit on full data for test set

    def _make_fold_map(self, X_col, y):
        means = y.groupby(X_col, dropna=False).mean()
        if self.smoothing > 0:
            counts = y.groupby(X_col, dropna=False).count()
            smooth = (counts * means + self.smoothing * y.mean()) / (counts + self.smoothing)
            means  = smooth
        return {k: r for r, k in enumerate(means.sort_values().index)}

    def fit(self, X, y):
        X, y = X.reset_index(drop=True), y.reset_index(drop=True)
        if self.cat_cols is None:
            self.cat_cols = X.select_dtypes(include='object').columns.tolist()

        kf = KFold(self.n_splits, shuffle=True, random_state=42)
        self.maps_ = {col: [None]*self.n_splits for col in self.cat_cols}

        for fold, (tr_idx, _) in enumerate(kf.split(X)):
            X_tr, y_tr = X.loc[tr_idx], y.loc[tr_idx]
            for col in self.cat_cols:
                self.maps_[col][fold] = self._make_fold_map(X_tr[col], y_tr)

        for col in self.cat_cols:
            self.global_map[col] = self._make_fold_map(X[col], y)

        return self

    def transform(self, X, y=None, fold=None):
        """
        • During CV pass fold index to use fold‑specific maps (leak‑free).
        • At inference time (fold=None) uses global map.
        """
        X = X.copy()
        tgt_maps = {col: (self.global_map[col] if fold is None else self.maps_[col][fold])
                    for col in self.cat_cols}
        for col, mapping in tgt_maps.items():
            X[col] = X[col].map(mapping).fillna(-1).astype(int)
        return X

In [ ]:
encode_stats = ['mean']
FOLDS          = 10
outer_kf       = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
oof            = np.zeros(len(train))
pred           = np.zeros(len(test))

for fold, (tr_idx, vl_idx) in enumerate(outer_kf.split(train), 1):
    print(f"--- Fold {fold} / {FOLDS} ---")

    X_tr_raw = train.loc[tr_idx, FEATURES].reset_index(drop=True)
    y_tr     = train.loc[tr_idx, TARGET].reset_index(drop=True)

    X_vl_raw = train.loc[vl_idx, FEATURES].reset_index(drop=True)
    y_vl     = train.loc[vl_idx, TARGET].reset_index(drop=True)

    X_ts_raw = test[FEATURES].copy()

    X_tr, X_vl, X_ts = X_tr_raw.copy(), X_vl_raw.copy(), X_ts_raw.copy()

    inner_kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
    for _, (in_tr_idx, in_vl_idx) in enumerate(inner_kf.split(X_tr_raw), 1):
        in_tr = pd.concat([X_tr_raw.loc[in_tr_idx], y_tr.loc[in_tr_idx]], axis=1)
        in_vl = X_tr_raw.loc[in_vl_idx].reset_index(drop=True)

        for col in encoded_columns:
            for stat in encode_stats:
                te_tmp = target_encode(
                    in_tr, in_vl.copy(),
                    col, TARGET,
                    stats=stat, prefix="TE"
                )
                te_col = f"TE_{col}_{stat}"
                X_tr.loc[in_vl_idx, te_col] = te_tmp[te_col].values

    tr_with_y = pd.concat([X_tr_raw, y_tr], axis=1)
    for col in encoded_columns:
        for stat in encode_stats:
            te_col = f"TE_{col}_{stat}"
            X_vl = target_encode(tr_with_y, X_vl,      col, TARGET,
                                  stats=stat, prefix="TE")
            X_ts = target_encode(tr_with_y, X_ts,      col, TARGET,
                                  stats=stat, prefix="TE")

    # 移除原本的交互類別特徵 (已經轉為 TE 數值特徵)
    X_tr.drop(encoded_columns, axis=1, inplace=True)
    X_vl.drop(encoded_columns, axis=1, inplace=True)
    X_ts.drop(encoded_columns, axis=1, inplace=True)    

    # 【修正重點】：只選取目前實際存在於 X_tr 中且為 object 或 category 的欄位
    current_cats = [col for col in CATS if col in X_tr.columns and X_tr[col].dtype.name in ['object', 'category']]

    enc = OrderedTargetEncoder(
        cat_cols=current_cats,
        n_splits=FOLDS,
        smoothing=20
    ).fit(X_tr, y_tr)

    X_tr[current_cats] = enc.transform(X_tr[current_cats], fold=None)[current_cats]
    X_vl[current_cats] = enc.transform(X_vl[current_cats], fold=None)[current_cats]
    X_ts[current_cats] = enc.transform(X_ts[current_cats], fold=None)[current_cats]
    
    model = XGBRegressor(
        tree_method='hist',
        max_depth=14,
        colsample_bytree=0.5,
        subsample=0.9,
        n_estimators=50_000,
        learning_rate=0.02,
        enable_categorical=True,
        min_child_weight=10,
        early_stopping_rounds=150,
        random_state = 42
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_vl, y_vl)],
        verbose=500
    )

    oof[vl_idx]  = model.predict(X_vl)
    pred        += model.predict(X_ts)

    del X_tr_raw, X_vl_raw, X_ts_raw, X_tr, X_vl, X_ts, y_tr, y_vl
    if fold != FOLDS:
        del model
    gc.collect()

pred /= FOLDS
rmse = mean_squared_error(train[TARGET], oof, squared=False)
print(f"Final OOF RMSE (XGB): {rmse:.5f}")

In [ ]:
importance_types = ['weight', 'gain', 'cover', 'total_gain', 'total_cover']

booster = model.get_booster()

for itype in importance_types:
    score = booster.get_score(importance_type=itype)
    score_series = pd.Series(score).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    score_series.head(30).plot(kind='bar')
    plt.title(f"Feature Importance - {itype}")
    plt.ylabel(itype)
    plt.xlabel("Features")
    plt.tight_layout()
    plt.show()

# Submission

In [ ]:
sub = pd.read_csv('/kaggle/input/playground-series-s5e4/sample_submission.csv')
sub[TARGET] = pred
sub.to_csv('submission.csv', index=False)
sub.head()

In [ ]:
import pickle

data_to_save = {
    'oof': oof,
    'pred': pred,
}

with open(f'oof_pred_{oof_pred_name}.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)


# with open('oof_pred.pkl', 'rb') as f:
#     data = pickle.load(f)

# oof = data['oof']
# pred = data['pred']